In [ ]:
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.colors as pc
from itertools import cycle
from Levenshtein import distance as levenshtein_distance
import os

# ==========================================
# 1. PARAMETERS
# ==========================================
PRUNING_THRESHOLD = 0.65  # Cut weak links (<65% similar)

FEATURE_FILE = r'F:\Resized_1024_Final\Macrolabel XyloNet Large SD1.xlsx'
BASE_CONN_FILE = r'F:\Resized_1024_Final\xylonet_base_space.xlsx'
LARGE_CONN_FILE = r'F:\Resized_1024_Final\xylonet_large_space.xlsx'

OUTPUT_BASE = r'F:\Resized_1024_Final\Xylonet_Base_Tight.html'
OUTPUT_LARGE = r'F:\Resized_1024_Final\Xylonet_Large_Spaced.html'

# ==========================================
# 2. LOAD DATA (MASTER DICTIONARIES)
# ==========================================
print("Loading Feature Data...")
try:
    df_features = pd.read_excel(FEATURE_FILE)
except Exception as e:
    print(f"Error loading feature file: {e}")
    exit()

df_features.columns = df_features.columns.str.strip().str.title()

def get_clean_id(name):
    return str(name).lower().replace(" ", "").replace("_", "").strip()

df_features['Clean_ID'] = df_features['Class'].apply(get_clean_id)
df_features['Combination'] = df_features['Combination'].fillna("").astype(str)

# Global Lookups
master_comb_dict = dict(zip(df_features['Clean_ID'], df_features['Combination']))
master_name_dict = dict(zip(df_features['Clean_ID'], df_features['Class']))
master_phase_dict = dict(zip(df_features['Clean_ID'], df_features['Phase']))

# ==========================================
# 3. MAP GENERATOR & REPORTER
# ==========================================
def generate_tuned_map(connection_file, output_filename, title, spacing_k):
    print(f"\n--- Processing: {title} ---")
    print(f"Applying Spacing Factor (k): {spacing_k}")
    
    try:
        df_conn = pd.read_excel(connection_file)
    except Exception as e:
        print(f"Skipping {title}: Could not load {connection_file} ({e})")
        return

    # --- STEP A: DEFINE THE UNIVERSE FOR THIS SPECIFIC MAP ---
    raw_node_a = set(df_conn['Node_A'].apply(get_clean_id))
    raw_node_b = set(df_conn['Node_B'].apply(get_clean_id))
    expected_nodes_in_this_map = raw_node_a.union(raw_node_b)
    
    print(f"Species found in input file: {len(expected_nodes_in_this_map)}")

    G = nx.Graph()
    for node_id in expected_nodes_in_this_map:
        if node_id in master_name_dict: 
            G.add_node(node_id)
    
    # --- STEP B: ADD EDGES ---
    for _, row in df_conn.iterrows():
        id_a = get_clean_id(row['Node_A'])
        id_b = get_clean_id(row['Node_B'])
        
        if id_a not in master_comb_dict or id_b not in master_comb_dict:
            continue
            
        comb_a = master_comb_dict[id_a]
        comb_b = master_comb_dict[id_b]
        
        dist = levenshtein_distance(comb_a, comb_b)
        max_len = max(len(comb_a), len(comb_b)) or 1
        similarity = 1 - (dist / max_len)
        
        if similarity < PRUNING_THRESHOLD:
            continue 
        
        cost = 1.0 - similarity
        G.add_edge(id_a, id_b, weight=similarity, cost=cost)

    print(f"Final Nodes in Graph: {len(G.nodes())}")

    # --- STEP C: CLUSTERS REPORT ---
    clusters = list(nx.connected_components(G))
    clusters.sort(key=len, reverse=True)
    
    color_pool = cycle(pc.qualitative.Dark24)
    cluster_colors = {}
    cluster_report_data = []
    
    for i, cluster in enumerate(clusters):
        assigned_color = next(color_pool)
        cluster_colors[tuple(cluster)] = assigned_color
        
        cluster_members = [master_name_dict[n] for n in cluster]
        cluster_report_data.append({
            'Cluster_ID': i + 1,
            'Color_Code': assigned_color,
            'Member_Count': len(cluster),
            'Members': ", ".join(cluster_members)
        })

    cluster_csv_name = output_filename.replace('.html', '_Clusters.csv')
    pd.DataFrame(cluster_report_data).to_csv(cluster_csv_name, index=False)
    print(f"Saved Cluster Report")

    # --- STEP D: NEIGHBOR REPORT (Direct 1-Step Only) ---
    print("Listing direct neighbors...")
    dist_report_data = []
    
    # 
    # We iterate over EDGES only. This guarantees 1-step connections.
    for u, v, data in G.edges(data=True):
        
        dist_report_data.append({
            'Node_A': master_name_dict[u],
            'Node_B': master_name_dict[v],
            'Distance': round(data['cost'], 4),        # 0.1 means close
            'Similarity': round(data['weight'], 4)     # 0.9 means similar
        })
            
    dist_csv_name = output_filename.replace('.html', '_Direct_Neighbors.csv')
    pd.DataFrame(dist_report_data).sort_values(by='Distance').to_csv(dist_csv_name, index=False)
    print(f"Saved Neighbor Report: {dist_csv_name}")

    # --- STEP E: LAYOUT & RENDER ---
    print("Calculating Physics Layout...")
    pos = nx.spring_layout(G, dim=3, weight='weight', k=spacing_k, iterations=1000, seed=42)
        
    fig = go.Figure()

    # Draw Lines
    for cluster_nodes, color in cluster_colors.items():
        if len(cluster_nodes) > 1:
            subgraph = G.subgraph(cluster_nodes)
            edge_x, edge_y, edge_z = [], [], []
            for u, v in subgraph.edges():
                x0, y0, z0 = pos[u]
                x1, y1, z1 = pos[v]
                edge_x.extend([x0, x1, None])
                edge_y.extend([y0, y1, None])
                edge_z.extend([z0, z1, None])
            fig.add_trace(go.Scatter3d(
                x=edge_x, y=edge_y, z=edge_z, mode='lines',
                line=dict(color=color, width=3), hoverinfo='none', name=f'Cluster {len(cluster_nodes)}'
            ))

    # Draw Nodes
    node_x = [pos[n][0] for n in G.nodes()]
    node_y = [pos[n][1] for n in G.nodes()]
    node_z = [pos[n][2] for n in G.nodes()]
    
    node_names = [master_name_dict[n] for n in G.nodes()]
    node_colors = [('#008B8B' if master_phase_dict[n] == 1 else '#8B008B') for n in G.nodes()]
    
    # Highlight Isolated Nodes with 'X'
    node_symbols = []
    for n in G.nodes():
        if G.degree(n) == 0:
            node_symbols.append('x') 
        else:
            node_symbols.append('square' if master_phase_dict[n] == 1 else 'circle')

    fig.add_trace(go.Scatter3d(
        x=node_x, y=node_y, z=node_z,
        mode='markers+text',
        marker=dict(size=5, color=node_colors, symbol=node_symbols, line=dict(width=1, color='black')),
        text=node_names, textfont=dict(color='black', size=10),
        textposition="top center", hoverinfo='text', name='Species'
    ))

    fig.update_layout(
        title=f"{title} (N={len(G.nodes())})",
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False), bgcolor='white'),
        paper_bgcolor='white', margin=dict(l=0, r=0, b=0, t=30), showlegend=False
    )
    
    fig.write_html(output_filename)
    print(f"Saved Visualization: {output_filename}")

# ==========================================
# 4. EXECUTE
# ==========================================

# XyloNet Base
generate_tuned_map(BASE_CONN_FILE, OUTPUT_BASE, "XyloNet Base (Tight)", spacing_k=7.5)

# XyloNet Large
generate_tuned_map(LARGE_CONN_FILE, OUTPUT_LARGE, "XyloNet Large (Spaced)", spacing_k=2.0)